# Heart Disease Classification with PySpark MLlib
Comprehensive Machine Learning Pipeline: Exploratory Data Analysis, Multi-Model Benchmarking, Hyperparameter Tuning, and Feature Importance.

### Workflow:
1. **Environment Setup & Data Ingestion**: SparkSession initialization and schema validation.
2. **Exploratory Data Analysis (EDA)**: Descriptive statistics, distribution plots, correlation matrix, target balance, and clinical risk factor breakdown.
3. **Data Preprocessing & Vectorization**: VectorAssembler and StandardScaler.
4. **Multi-Model Machine Learning Benchmark**:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
   - GBT (Gradient-Boosted Trees) Classifier
5. **Model Evaluation & Confusion Matrix**: Multi-metric evaluation (ROC-AUC, Accuracy, Precision, Recall, F1-Score).
6. **Hyperparameter Tuning**: 5-Fold Cross-Validation on top ensemble model.
7. **Feature Importance Interpretation**: Ranking clinical indicators driving heart disease risk.

In [ ]:
# 1. Imports and Spark Initialization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, avg, desc
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression, 
    DecisionTreeClassifier, 
    RandomForestClassifier, 
    GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Setup styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

spark = SparkSession.builder \
    .appName("HeartDiseaseClassification_Advanced") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## 2. Data Ingestion & Schema Exploration
Load `heart.csv` into a PySpark DataFrame with schema inference enabled.

In [ ]:
# Load dataset
data_path = "heart.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

print(f"Dataset Dimensions: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()
df.show(5, truncate=False)

### Check Missing Values & Statistical Summary

In [ ]:
# Check for null / NaN values in every column
missing_counts = df.select([
    count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in df.columns
])
print("Missing values per column:")
missing_counts.show()

# Descriptive statistics
df.describe().toPandas().transpose()

## 3. Exploratory Data Analysis (EDA)
Convert aggregated/sample data to Pandas for visualization of distributions, correlation, and demographic risk factors.

In [ ]:
# Convert to Pandas for visualization
pdf = df.toPandas()

# 1. Target Class Distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

target_counts = pdf['target'].value_counts()
ax[0].pie(target_counts, labels=['Disease (1)', 'No Disease (0)'], autopct='%1.1f%%', 
          startangle=90, colors=['#e74c3c', '#3498db'], explode=(0.05, 0))
ax[0].set_title("Target Class Distribution (Heart Disease)")

sns.countplot(data=pdf, x='target', hue='sex', palette='Set2', ax=ax[1])
ax[1].set_title("Heart Disease Count by Sex (0 = Female, 1 = Male)")
ax[1].set_xticklabels(['No Disease', 'Disease'])
plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation Matrix Heatmap
plt.figure(figsize=(12, 8))
corr = pdf.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, linewidths=0.5)
plt.title("Correlation Matrix of Clinical Attributes", fontsize=14, weight='bold')
plt.show()

In [ ]:
# 3. Distributions of Key Quantitative Indicators by Disease Status
features_to_plot = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    sns.kdeplot(data=pdf, x=feature, hue='target', fill=True, common_norm=False, 
                palette=['#3498db', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f"Distribution of {feature} by Diagnosis", weight='bold')
    axes[i].legend(['Disease', 'No Disease'])

# Hide extra subplot
axes[5].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Impact of Chest Pain Type (cp) & Thal on Target
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=pdf, x='cp', hue='target', palette=['#3498db', '#e74c3c'], ax=ax[0])
ax[0].set_title("Chest Pain Type (cp) vs Diagnosis", weight='bold')
ax[0].set_xlabel("Chest Pain Type (0: Typical, 1: Atypical, 2: Non-anginal, 3: Asymptomatic)")
ax[0].legend(['No Disease', 'Disease'])

sns.countplot(data=pdf, x='thal', hue='target', palette=['#3498db', '#e74c3c'], ax=ax[1])
ax[1].set_title("Thalassemia (thal) vs Diagnosis", weight='bold')
ax[1].set_xlabel("Thal (0: Normal, 1: Fixed Defect, 2: Reversible Defect)")
ax[1].legend(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

## 4. Feature Pipeline & Train/Test Split
Assemble all 13 predictor columns into a feature vector and apply standard scaling.

In [ ]:
feature_cols = [c for c in df.columns if c != 'target']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

# Train/Test Split (80/20) with deterministic seed
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Instances: {train_data.count()}")
print(f"Testing Instances:  {test_data.count()}")

## 5. Multi-Model Benchmark & Evaluation
We benchmark four classification architectures:
1. **Logistic Regression (Baseline)**
2. **Decision Tree Classifier**
3. **Random Forest Classifier**
4. **Gradient-Boosted Trees (GBT)**

In [ ]:
# Define standard evaluators
evaluator_roc = BinaryClassificationEvaluator(labelCol="target", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="target", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="target", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="target", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="target", metricName="f1")

def evaluate_predictions(predictions, model_name="Model"):
    roc_auc = evaluator_roc.evaluate(predictions)
    acc = evaluator_acc.evaluate(predictions)
    prec = evaluator_prec.evaluate(predictions)
    rec = evaluator_rec.evaluate(predictions)
    f1 = evaluator_f1.evaluate(predictions)
    
    return {
        "Model": model_name,
        "ROC-AUC": round(roc_auc, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    }

def get_confusion_matrix(predictions):
    y_true = [int(r['target']) for r in predictions.select('target').collect()]
    y_pred = [int(r['prediction']) for r in predictions.select('prediction').collect()]
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm

In [ ]:
# Instantiate Classifiers
lr = LogisticRegression(featuresCol="features", labelCol="target", maxIter=20)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="target", seed=42)
rf = RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=50, seed=42)
gbt = GBTClassifier(featuresCol="features", labelCol="target", maxIter=30, seed=42)

models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "Gradient-Boosted Trees": gbt
}

trained_models = {}
predictions_dict = {}
benchmark_results = []

for name, clf in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(stages=[assembler, scaler, clf])
    model = pipeline.fit(train_data)
    preds = model.transform(test_data)
    
    trained_models[name] = model
    predictions_dict[name] = preds
    metrics = evaluate_predictions(preds, name)
    benchmark_results.append(metrics)

# Display comparison DataFrame
results_df = pd.DataFrame(benchmark_results)
results_df

In [ ]:
# Plot Confusion Matrix for each model side-by-side
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for idx, (name, preds) in enumerate(predictions_dict.items()):
    cm = get_confusion_matrix(preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['No Disease (0)', 'Disease (1)'],
                yticklabels=['No Disease (0)', 'Disease (1)'])
    axes[idx].set_title(f"{name} - Confusion Matrix", weight='bold')
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning with 5-Fold Cross-Validation
We tune the Random Forest ensemble using `CrossValidator` across tree counts and maximum depth to maximize generalization.

In [ ]:
# Build parameter grid for Random Forest
rf_tune = RandomForestClassifier(featuresCol="features", labelCol="target", seed=42)
tune_pipeline = Pipeline(stages=[assembler, scaler, rf_tune])

param_grid = ParamGridBuilder() \
    .addGrid(rf_tune.numTrees, [30, 60, 100]) \
    .addGrid(rf_tune.maxDepth, [4, 6, 8]) \
    .build()

cv = CrossValidator(
    estimator=tune_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator_roc,
    numFolds=5,
    seed=42
)

print("Executing 5-Fold Cross-Validation...")
cv_model = cv.fit(train_data)
best_rf_preds = cv_model.transform(test_data)

tuned_metrics = evaluate_predictions(best_rf_preds, "Tuned Random Forest (CV)")
print("Tuned Model Evaluation:")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v}")

best_rf_stage = cv_model.bestModel.stages[-1]
print(f"Best numTrees: {best_rf_stage.getNumTrees}")
print(f"Best maxDepth: {best_rf_stage.getOrDefault('maxDepth')}")

## 7. Feature Importance Analysis
Analyzing the relative importance of clinical attributes extracted from the tuned ensemble model.

In [ ]:
# Extract feature importances
importances = best_rf_stage.featureImportances.toArray()
feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', palette='viridis')
plt.title("Clinical Feature Importance for Heart Disease Prediction (Random Forest)", fontsize=13, weight='bold')
plt.xlabel("Importance Score")
plt.ylabel("Attribute")
plt.show()

feat_imp_df

## 8. Summary of Results & Conclusions

In [ ]:
# Add tuned model to final summary
final_summary_df = pd.concat([results_df, pd.DataFrame([tuned_metrics])], ignore_index=True)
final_summary_df = final_summary_df.sort_values(by='ROC-AUC', ascending=False).reset_index(drop=True)

print("=== FINAL MODEL PERFORMANCE BENCHMARK ===")
print(final_summary_df.to_string(index=False))

# Identify top predictor attributes
top_features = feat_imp_df.head(3)['Feature'].tolist()
print(f"\nKey Finding: The most predictive risk markers identified are: {', '.join(top_features)}.")

In [ ]:
# Stop Spark Session
spark.stop()
print("Spark Session successfully terminated.")